# R6. AI data-analyst bake-off (35 min)  |  Client brief: **Economics and sociology**
> *"Here is our regional household survey. Please clean it and give me three findings. I'm sure trust in institutions is rising everywhere."*

You may use **any tool**: ChatGPT (upload the CSV), Copilot, Gemini in Colab, or any other assistant. This notebook only (1) tells you your arm, (2) lets you check your tool's claims, and (3) reports your numbers.
The data file is `coding_tools/survey_messy.csv`. The **answer key is with the instructor**.

**How the arena works**
- Your prompt can use placeholders: `{text}` (the item), `{labels}`, `{codebook}` (the official definitions), `{schema}` (extraction).
- The harness adds one output rule so everyone is parsed the same way. You do **not** need to ask for a specific format.
- Every run costs real money from your allowance and appears on the board. **Write your hypothesis in `note=` before you run.**
- The hidden test set stays with the instructor. Scores you see here are on the *dev* set.

In [ ]:
# ===== Run this cell first =====
import os, sys, hashlib, json
KIT_URL = "https://github.com/atsizelti/prompt_engineering.git"   # public repo with the kit
KIT_DIR = "arena_kit/prompt_arena_kit"                            # folder inside the clone
import importlib.util
if os.path.exists("arena_kit"):
    os.system("git -C arena_kit pull -q")                # always refresh a copy cloned earlier in this session
elif importlib.util.find_spec("arena") is None:
    os.system(f"git clone -q {KIT_URL} arena_kit")
if os.path.exists(KIT_DIR) and KIT_DIR not in sys.path:
    sys.path.insert(0, KIT_DIR)
import arena

NICK      = "your-nickname"      # pick a nickname (no real name needed)
PROVIDER  = "openai"             # "openai", or "mock" (practice with no key)
ENDPOINT  = "https://script.google.com/macros/s/AKfycbxdVYynFuPHaHg73kTFzL5R0TYXrPA8Tr0adMEsb0LKUS5vpDgysktWLPTTcLWrS7Jz/exec"     # the class board (already filled in: you do not need to change it)
TOKEN     = "f-BEqiAfHhAc2O-b"
ALLOWANCE = 0.0                # dollars you may spend (the whole class shares one small budget; re-running this cell does not reset it)

if os.environ.get("ARENA_MOCK"):
    PROVIDER = "mock"
if os.environ.get("ARENA_NO_BOARD"):                  # used by the instructor's offline tests
    ENDPOINT = TOKEN = ""
KEY = None
if PROVIDER != "mock":
    try:
        from google.colab import userdata
        KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        KEY = os.environ.get("OPENAI_API_KEY")
try:                                                  # the board address and password, set once as Colab secrets
    from google.colab import userdata
    ENDPOINT = ENDPOINT or userdata.get("BOARD_ENDPOINT")
    TOKEN = TOKEN or userdata.get("BOARD_TOKEN")
except Exception:
    ENDPOINT = ENDPOINT or os.environ.get("BOARD_ENDPOINT", "")
    TOKEN = TOKEN or os.environ.get("BOARD_TOKEN", "")
ENDPOINT, TOKEN = "".join(ENDPOINT.split()), "".join(TOKEN.split())   # remove hidden spaces and line breaks from pasting
arena.setup(NICK, provider=PROVIDER, api_key=KEY, endpoint=ENDPOINT or None,
            token=TOKEN or None, allowance_usd=ALLOWANCE)
arena.set_round("R6")

In [ ]:
ARMS = ["none", "house-rules", "house-rules+check"]
ARM = ARMS[int(hashlib.md5(("R6" + NICK).encode()).hexdigest(), 16) % 3]
print("Your arm:", ARM)

HOUSE_RULES = """HOUSE RULES for this analysis
- Before analysing, look for duplicate respondents, impossible values, and missing-value codes.
- State every assumption you make (dates, currency, how you treat missing values).
- Never silently drop rows; report how many you changed or removed and why.
- Report results by region and by year, not only pooled."""
CHECK = """
BEFORE you report results: write assertions (row counts, value ranges, no duplicate IDs, date range) and SHOW their output.
Do not say "looks good" without evidence."""

if ARM == "none":
    print("Use no special instructions: just paste the client brief.")
elif ARM == "house-rules":
    print("Paste this into custom instructions or your first message:\n\n" + HOUSE_RULES)
else:
    print("Paste this into custom instructions or your first message:\n\n" + HOUSE_RULES + CHECK)

### Verify, do not trust
Below is a scratch area. Whatever your tool tells you, **check it yourself** in at least one place.
Load the file and test one claim.

In [ ]:
import pandas as pd
CSV = next((p for p in ["coding_tools/survey_messy.csv", "arena_kit/prompt_arena_kit/coding_tools/survey_messy.csv", "survey_messy.csv"]
            if os.path.exists(p)), None)      # adjust the path if you uploaded the file elsewhere
try:
    df = pd.read_csv(CSV)
    print(df.shape); display(df.head())
except Exception as e:
    print("Could not load:", e)

In [ ]:
# Your own evidence cell: one assertion or count that tests a claim your tool made.
# Example (edit): print(df["respondent_id"].duplicated().sum(), "repeated IDs")

### Report your numbers (2 minutes)
`traps_caught` is out of 6; count it against the key the instructor will read out at the end (leave it for the end of the round).

In [ ]:
# Run this at the END of the round.
TOOL = "ChatGPT"          # what you used
TRAPS_CAUGHT = 0          # out of 6, after the instructor reveals the key
TURNS = 0                 # number of back-and-forth messages
MODE = "fast"             # "fast" or "extended-thinking"
VERIFIED = False          # did you verify at least one claim yourself?
arena.report_r6(TOOL, TRAPS_CAUGHT, TURNS, MODE, VERIFIED, arm=ARM)